# 10 — Executive Brief Generator

> **A concise, evidence-backed weekly brief built from the computed numbers only.**

| | |
|---|---|
| **Audience** | Executive sponsors, weekly read-out attendees |
| **Owner** | CX Complaint Intelligence team |
| **Status** | Reference implementation (portfolio sample) |
| **Last reviewed** | 2026-05-25 |
| **Runtime** | ~10 s on the bundled sample |
| **Reproduces from** | `reports/sample/complaint_scores.csv + reports/sample/topic_terms.csv` |

## Business question

Can we produce a stakeholder-ready Markdown brief whose numbers come exclusively from computed tables, never from prose the model might hallucinate?

## How to read this notebook

1. Section **0** initialises the environment and loads data.
2. Sections **1-N** build the analysis tables and visuals.
3. The **Insights** section is generated from the computed tables &mdash; it does not contain hand-written claims.
4. The **Limitations** section flags known caveats before stakeholders act on the numbers.
5. The **Exports** section persists every table and figure under `reports/notebook/10/` so the output is auditable.


## 0. Setup

This cell:

- locates the project root from wherever the kernel was started;
- applies the shared matplotlib house style and pandas display options;
- creates the artefact directory `reports/notebook/10/`;
- captures the Python and library versions used for this run.


In [1]:
from __future__ import annotations

import warnings
from IPython.display import Markdown, display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cx_complaints.reports.notebook_utils import (
    PALETTE,
    RISK_COLORS,
    concentration_table,
    horizontal_bar,
    kpi_table,
    load_complaints,
    load_pipeline_outputs,
    monthly_summary,
    render_environment,
    save_figure,
    save_table,
    setup_notebook,
    topic_labels,
    write_environment_snapshot,
)

warnings.filterwarnings("ignore", category=FutureWarning)

ctx = setup_notebook("10")
render_environment(ctx)


,field,value
0,python,3.12.11
1,platform,Windows-11-10.0.26200-SP0
2,numpy,2.4.6
3,pandas,2.3.3
4,matplotlib,3.10.9
5,scikit-learn,1.8.0
6,project_root,C:\Users\diogo\work_code\portfolio\cx-complain...
7,notebook_id,10


## 1. Load pipeline outputs

In [2]:
scores, topic_terms = load_pipeline_outputs(ctx)
scores = scores.dropna(subset=["date_received"]).copy()
scores["week"] = scores["date_received"].dt.to_period("W").astype(str)
labels = topic_labels(topic_terms, top_n=4)
print(f"Scored complaints: {len(scores):,}")

Scored complaints: 1,000


C:\Users\diogo\AppData\Local\Temp\ipykernel_46808\923331498.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  scores["week"] = scores["date_received"].dt.to_period("W").astype(str)


## 2. Build weekly KPI context

In [3]:
weekly = (
    scores.groupby("week", as_index=False)
    .agg(
        complaints=("complaint_id", "size"),
        high_risk=("risk_level", lambda s: (s == "high").sum()),
        avg_risk_score=("risk_score", "mean"),
        untimely=("timely", lambda s: (s.fillna("").str.lower() != "yes").sum()),
    )
    .sort_values("week")
)
weekly["high_risk_share_pct"] = (weekly["high_risk"] / weekly["complaints"] * 100).round(1)
weekly["untimely_share_pct"] = (weekly["untimely"] / weekly["complaints"] * 100).round(1)
weekly["avg_risk_score"] = weekly["avg_risk_score"].round(2)
weekly.tail(4)

,week,complaints,high_risk,avg_risk_score,untimely,high_risk_share_pct,untimely_share_pct
1,2024-06-03/2024-06-09,260,27,3.32,0,10.40,0.00
2,2024-06-10/2024-06-16,224,34,3.54,1,15.20,0.40
3,2024-06-17/2024-06-23,243,43,3.57,0,17.70,0.00
4,2024-06-24/2024-06-30,236,29,3.48,0,12.30,0.00


## 3. Compute topic and priority-queue context

In [4]:
observed_weeks = scores["week"].nunique()
if observed_weeks < 2:
    raise ValueError(
        "Executive brief requires at least two weeks of data. "
        "Run the pipeline against a multi-week fetch."
    )

current = weekly.iloc[-1]
previous = weekly.iloc[-2]

delta = {
    "complaints": int(current["complaints"] - previous["complaints"]),
    "high_risk_share_pct": round(current["high_risk_share_pct"] - previous["high_risk_share_pct"], 1),
    "untimely_share_pct": round(current["untimely_share_pct"] - previous["untimely_share_pct"], 1),
    "avg_risk_score": round(current["avg_risk_score"] - previous["avg_risk_score"], 2),
}

topic_weekly = (
    scores.dropna(subset=["topic_id"])
    .assign(topic_id=lambda d: d["topic_id"].astype(int))
    .groupby(["week", "topic_id"], as_index=False).size().rename(columns={"size": "complaints"})
)
latest = topic_weekly[topic_weekly["week"] == current["week"]][["topic_id", "complaints"]].rename(columns={"complaints": "latest_complaints"})
prev = topic_weekly[topic_weekly["week"] == previous["week"]][["topic_id", "complaints"]].rename(columns={"complaints": "prev_complaints"})
topic_move = (
    labels.merge(latest, on="topic_id", how="left").merge(prev, on="topic_id", how="left")
)
topic_move[["latest_complaints", "prev_complaints"]] = (
    topic_move[["latest_complaints", "prev_complaints"]].fillna(0).astype(int)
)
topic_move["delta_complaints"] = topic_move["latest_complaints"] - topic_move["prev_complaints"]
topic_move = topic_move.sort_values("delta_complaints", ascending=False).reset_index(drop=True)

queue = scores.copy()
queue["timely_is_yes"] = queue["timely"].fillna("").astype(str).str.lower() == "yes"
queue["weak_response"] = queue["company_response"].fillna("").astype(str).str.lower().isin(
    ["closed", "closed with explanation", "in progress"]
)
priority_queue = queue[
    (queue["risk_level"] == "high")
    & (~queue["timely_is_yes"])
    & (queue["weak_response"])
]
priority_queue_week = priority_queue[priority_queue["week"] == current["week"]]
queue_stats = {
    "priority_queue_count": int(len(priority_queue_week)),
    "priority_queue_share_pct": round(
        len(priority_queue_week) / max(int(current["complaints"]), 1) * 100, 1,
    ),
}
queue_stats

{'priority_queue_count': 0, 'priority_queue_share_pct': 0.0}

## 4. Generate the Markdown brief (numbers only from tables)

In [5]:
top_rising = topic_move.head(3)
topic_lines = [
    f"- Topic {int(row['topic_id'])} ({row['topic_label']}): "
    f"{int(row['delta_complaints']):+,} complaints WoW (now {int(row['latest_complaints']):,}, was {int(row['prev_complaints']):,})."
    for _, row in top_rising.iterrows()
]

brief_lines = [
    f"# Weekly Executive Brief ({current['week']})",
    "",
    "## What changed",
    "",
    f"- Complaints: {int(current['complaints']):,} ({delta['complaints']:+,} WoW).",
    f"- High-risk share: {current['high_risk_share_pct']:.1f}% ({delta['high_risk_share_pct']:+.1f} pp WoW).",
    f"- Untimely share: {current['untimely_share_pct']:.1f}% ({delta['untimely_share_pct']:+.1f} pp WoW).",
    f"- Average risk score: {current['avg_risk_score']:.2f} ({delta['avg_risk_score']:+.2f} WoW).",
    "",
    "## Emerging topics",
    "",
    *topic_lines,
    "",
    "## Operational queue",
    "",
    f"- Priority queue (high-risk + untimely + weak response): "
    f"{queue_stats['priority_queue_count']:,} complaints "
    f"({queue_stats['priority_queue_share_pct']:.1f}% of weekly volume).",
    "",
    "## Recommended actions",
    "",
    "1. Review the top priority-queue items within 24 hours.",
    "2. Assign topic owners and validate root causes for the rising topics.",
    "3. Track whether the high-risk and untimely shares normalise next week.",
    "",
    "## Data sources",
    "",
    "- `reports/sample/complaint_scores.csv` &mdash; per-complaint risk and topic assignments.",
    "- `reports/sample/topic_terms.csv` &mdash; top terms per discovered topic.",
    "- Pipeline: `cx-complaints analyze --input data/raw/cfpb_sample.csv --output-dir reports/sample`.",
]
brief_text = "\n".join(brief_lines)
display(Markdown(brief_text))

# Weekly Executive Brief (2024-06-24/2024-06-30)

## What changed

- Complaints: 236 (-7 WoW).
- High-risk share: 12.3% (-5.4 pp WoW).
- Untimely share: 0.0% (+0.0 pp WoW).
- Average risk score: 3.48 (-0.09 WoW).

## Emerging topics

- Topic 4 (credit, report, credit report, information): +6 complaints WoW (now 103, was 97).
- Topic 0 (redacted, redacted redacted, money, account): +5 complaints WoW (now 76, was 71).
- Topic 3 (payment, late, payments, creditor): +3 complaints WoW (now 21, was 18).

## Operational queue

- Priority queue (high-risk + untimely + weak response): 0 complaints (0.0% of weekly volume).

## Recommended actions

1. Review the top priority-queue items within 24 hours.
2. Assign topic owners and validate root causes for the rising topics.
3. Track whether the high-risk and untimely shares normalise next week.

## Data sources

- `reports/sample/complaint_scores.csv` &mdash; per-complaint risk and topic assignments.
- `reports/sample/topic_terms.csv` &mdash; top terms per discovered topic.
- Pipeline: `cx-complaints analyze --input data/raw/cfpb_sample.csv --output-dir reports/sample`.

## 5. Insights

In [6]:
rising_count = int((topic_move["delta_complaints"] > 0).sum())
biggest_mover = topic_move.head(1).iloc[0]
insights = (
    f"- Rising topics this week: **{rising_count}** of {len(topic_move)} topics increased.\n"
    f"- Biggest mover: Topic {int(biggest_mover['topic_id'])} "
    f"({biggest_mover['topic_label']}) with delta {int(biggest_mover['delta_complaints']):+,} complaints.\n"
    f"- Priority queue is {queue_stats['priority_queue_share_pct']:.1f}% of this week’s volume."
)
display(Markdown(insights))

- Rising topics this week: **4** of 8 topics increased.
- Biggest mover: Topic 4 (credit, report, credit report, information) with delta +6 complaints.
- Priority queue is 0.0% of this week’s volume.

## Limitations

- The risk score is heuristic, not a legal or regulatory finding. It exists to triage analyst attention.
- CFPB narratives are only published when the consumer consents, so coverage is partial.
- Company-level comparisons mix companies of very different sizes &mdash; always pair them with volume context.
- The sample bundled with this repo is a small subset of the public CFPB dataset and is meant for demonstration, not production decisions.

## Exports

All tables and figures generated above are saved under `reports/notebook/10/`. The environment snapshot is written alongside so a reader can reproduce the run.


In [7]:
(ctx.artefact_dir / "weekly_executive_brief.md").write_text(brief_text, encoding="utf-8")
save_table(ctx, weekly, "weekly_scorecard")
save_table(ctx, topic_move, "topic_weekly_movement")
pd.DataFrame([queue_stats]).to_csv(ctx.artefact_dir / "queue_stats.csv", index=False)
write_environment_snapshot(ctx)
sorted(ctx.artefact_dir.iterdir())

[WindowsPath('C:/Users/diogo/work_code/portfolio/cx-complaint-intelligence/reports/notebook/10/environment.json'),
 WindowsPath('C:/Users/diogo/work_code/portfolio/cx-complaint-intelligence/reports/notebook/10/queue_stats.csv'),
 WindowsPath('C:/Users/diogo/work_code/portfolio/cx-complaint-intelligence/reports/notebook/10/topic_weekly_movement.csv'),
 WindowsPath('C:/Users/diogo/work_code/portfolio/cx-complaint-intelligence/reports/notebook/10/weekly_executive_brief.md'),
 WindowsPath('C:/Users/diogo/work_code/portfolio/cx-complaint-intelligence/reports/notebook/10/weekly_scorecard.csv')]